# ADS v2: Motor in-life stand-ins

Creates the 44 event family tables that `EHN_ADS_v2_15_Motor_InLife` would build, **empty**, so that `EHN_ADS_v2_19_EventStream` can run while 15_Motor_InLife is left out (its arrears families read `ods_v2.PolicyLoanStatus`, which is not built yet). It does the same as `skip = "15_Motor_InLife"` in `EHN_ADS_v2_Run`.

A table is created only where it does not exist already: a table left by an earlier run of 15_Motor_InLife is kept as it is, with its events. `ads_v2.AdsFamilyCheck` reports the empty families as "no events". Once 15_Motor_InLife runs again, its own tables replace these and this notebook is no longer needed.

In [ ]:
%%pyspark
# Parameters
stg_schema = "stg"   # stg_schema in EHN_ADS_v2_00_Setup


In [ ]:
%%pyspark
import json
TABLES = json.loads(r"""["adsv2_evt_m1","adsv2_evt_m1_f1","adsv2_evt_m_v1","adsv2_evt_m_v2","adsv2_evt_m_v3","adsv2_evt_m_v4","adsv2_evt_m3","adsv2_evt_m4","adsv2_evt_m4a","adsv2_evt_m6","adsv2_evt_m3_b1_1","adsv2_evt_m3_b1_2","adsv2_evt_m3_b1_4","adsv2_evt_m3_b1_5","adsv2_evt_m3_b1_3","adsv2_evt_m3_b1_6","adsv2_evt_c1a","adsv2_evt_c1b","adsv2_evt_c1a_f1","adsv2_evt_c2","adsv2_evt_c3","adsv2_evt_c4","adsv2_evt_c5","adsv2_evt_c6","adsv2_evt_arr_1","adsv2_evt_arr_2","adsv2_evt_arr_o1","adsv2_evt_arr_3","adsv2_evt_arr_o2","adsv2_evt_cl1a","adsv2_evt_cl1b","adsv2_evt_cl1c","adsv2_evt_cl3_f1","adsv2_evt_cl4","adsv2_evt_cl_o1","adsv2_evt_cl_o2","adsv2_evt_d1","adsv2_evt_d1_f1","adsv2_evt_d3_f1","adsv2_evt_d_v2","adsv2_evt_d_v3","adsv2_evt_d_v4","adsv2_evt_d_v5","adsv2_evt_d_v6"]""")
EVENT_COLUMNS = ("EventId bigint, EventTypeId string, EventDescription string, PolicyTypeGroup string, Grain string, "
                 "EventDateTime timestamp, EventDate date, EventTimeBasis string, SourceSystemId int, "
                 "EventSourceReference string, SourcePolicyReference string, SourceQuoteReference string, "
                 "SourceCustomerReference string, SourceCustomerReferenceType string, PolicyId bigint, QuoteId bigint, "
                 "AdsBatchId string")
created = 0
for t in TABLES:
    if not spark.catalog.tableExists(f"{stg_schema}.{t}"):
        spark.sql(f"create table {stg_schema}.{t} ({EVENT_COLUMNS})")
        created += 1
print(f"{created} empty stand-in family tables created, {len(TABLES) - created} already existed and were left as they are")
